# Notebook 1: Data Loading & Cost Analysis
Loads all processed CSVs, validates data quality, and produces cost-by-tier charts.
Outputs: `cost_by_tier_bar.png`, `career_cost_by_tier.png`, `career_cost_by_tier.csv`, `cost_components_by_tier.png`, `socal_club_cost.png`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA = '../data/processed/'

programs = pd.read_csv(DATA + 'programs.csv')
costs    = pd.read_csv(DATA + 'costs.csv')
schols   = pd.read_csv(DATA + 'scholarship_outcomes.csv')
pros     = pd.read_csv(DATA + 'pro_outcomes.csv')
demo     = pd.read_csv(DATA + 'demographics.csv')

for name, df in [('programs', programs), ('costs', costs),
                 ('scholarships', schols), ('pro_outcomes', pros),
                 ('demographics', demo)]:
    print(f'\n=== {name} ===')
    print(f'  Shape: {df.shape}  |  Columns: {list(df.columns)}')

In [ ]:
# Null check across all datasets
for name, df in [('programs', programs), ('costs', costs),
                 ('scholarships', schols), ('pro_outcomes', pros),
                 ('demographics', demo)]:
    nulls = df.isnull().sum()
    if nulls.any():
        print(f'Nulls in {name}:')
        print(nulls[nulls > 0])
    else:
        print(f'{name}: no nulls')

# Confirm cost totals are internally consistent
cost_cols = ['annual_club_fee_mid','equipment_est','tournament_fees_local',
             'tournament_fees_regional','tournament_fees_national',
             'travel_est_local','travel_est_regional','travel_est_national',
             'coaching_extra','camp_fees','college_id_camps']
costs['computed_total_mid'] = costs[cost_cols].sum(axis=1)
costs['total_check_diff'] = costs['computed_total_mid'] - costs['total_mid']
bad = costs[costs['total_check_diff'].abs() > 1]
print(f'\nCost total discrepancies: {len(bad)} rows (should be 0)')

## Annual Cost by Tier

In [ ]:
costs = costs.merge(programs[['program_id','name']], on='program_id', how='left')

tier_cost = costs.groupby('tier')[['total_low','total_mid','total_high']].mean().round(0)
print('Mean annual cost by tier:')
print(tier_cost)

tier_cost.plot(kind='bar', color=['#2ecc71','#3498db','#e74c3c'], alpha=0.85)
plt.title('Average Annual Cost by Program Tier (Low / Mid / High)', fontsize=14)
plt.xlabel('Tier (1=Recreational → 4=Pro Academy)')
plt.ylabel('Annual Cost (USD)')
plt.xticks(rotation=0)
plt.legend(['Low Estimate','Mid Estimate','High Estimate'])
plt.tight_layout()
plt.savefig('../data/analysis/cost_by_tier_bar.png', dpi=150)
plt.show()

## Career Cost (U6–U18)
Career years: Tier 1 = 13 yrs (U6+), Tier 2 = 9 yrs (U10+), Tier 3/4 = 7 yrs (U13+).

In [ ]:
career_years = {1: 13, 2: 9, 3: 7, 4: 7}

career_cost = []
for tier, years in career_years.items():
    row = tier_cost.loc[tier]
    career_cost.append({
        'tier': tier,
        'years': years,
        'career_low':  row['total_low']  * years,
        'career_mid':  row['total_mid']  * years,
        'career_high': row['total_high'] * years,
    })

career_df = pd.DataFrame(career_cost).set_index('tier')
career_df.to_csv('../data/analysis/career_cost_by_tier.csv')
print('Career total cost by tier:')
print(career_df)

fig, ax = plt.subplots()
x = career_df.index
ax.bar(x - 0.25, career_df['career_low'],  width=0.25, label='Low',  color='#2ecc71', alpha=0.85)
ax.bar(x,         career_df['career_mid'],  width=0.25, label='Mid',  color='#3498db', alpha=0.85)
ax.bar(x + 0.25, career_df['career_high'], width=0.25, label='High', color='#e74c3c', alpha=0.85)
ax.set_title('Total Career Cost by Tier (U6–U18)', fontsize=14)
ax.set_xlabel('Tier')
ax.set_ylabel('Total Career Cost (USD)')
ax.set_xticks(x)
ax.set_xticklabels(['Tier 1\nRec', 'Tier 2\nTravel', 'Tier 3\nECNL/MLS NEXT', 'Tier 4\nPro Academy'])
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))
plt.tight_layout()
plt.savefig('../data/analysis/career_cost_by_tier.png', dpi=150)
plt.show()

## Cost Component Breakdown

In [ ]:
cost_components = ['annual_club_fee_mid','equipment_est','tournament_fees_local',
                   'tournament_fees_regional','tournament_fees_national',
                   'travel_est_local','travel_est_regional','travel_est_national',
                   'coaching_extra','camp_fees','college_id_camps']

breakdown = costs.groupby('tier')[cost_components].mean().round(0)
breakdown.T.plot(kind='bar', stacked=False)
plt.title('Average Cost Component by Tier', fontsize=13)
plt.ylabel('USD')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Tier', loc='upper right')
plt.tight_layout()
plt.savefig('../data/analysis/cost_components_by_tier.png', dpi=150)
plt.show()

# Stacked proportion — what % of cost is travel?
breakdown_pct = breakdown.div(breakdown.sum(axis=1), axis=0) * 100
breakdown_pct.T.plot(kind='bar', stacked=True, colormap='tab10')
plt.title('Cost Component Share by Tier (%)', fontsize=13)
plt.ylabel('% of Total')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../data/analysis/cost_pct_breakdown.png', dpi=150)
plt.show()

## SoCal Club Comparison

In [ ]:
socal_clubs = ['San Diego Surf','Strikers FC','Legends FC','Pateadores',
               'LA Galaxy Academy','LAFC Academy']
socal = costs[costs['name'].isin(socal_clubs)].copy()

if not socal.empty:
    fig, ax = plt.subplots(figsize=(10,5))
    bar_w = 0.25
    clubs = socal['name'].unique()
    x = np.arange(len(clubs))
    ax.bar(x - bar_w, socal.groupby('name')['total_low'].mean(),  bar_w, label='Low',  color='#2ecc71')
    ax.bar(x,          socal.groupby('name')['total_mid'].mean(),  bar_w, label='Mid',  color='#3498db')
    ax.bar(x + bar_w, socal.groupby('name')['total_high'].mean(), bar_w, label='High', color='#e74c3c')
    ax.set_xticks(x)
    ax.set_xticklabels(clubs, rotation=20, ha='right')
    ax.set_title('Annual Cost by SoCal Club', fontsize=13)
    ax.set_ylabel('USD')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.legend()
    plt.tight_layout()
    plt.savefig('../data/analysis/socal_club_cost.png', dpi=150)
    plt.show()
else:
    print('No SoCal club records matched — check program names in costs.csv')